In [17]:
# ============================================================
# CASE 7: Bagging Variants with Imbalance-Aware Sampling
# ============================================================

import pandas as pd
import numpy as np
import time
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestClassifier, BaggingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score,
                             recall_score, f1_score, matthews_corrcoef,
                             classification_report, confusion_matrix)

# Imbalanced-learn models
from imblearn.ensemble import (BalancedRandomForestClassifier,
                                EasyEnsembleClassifier,
                                RUSBoostClassifier)

print("All libraries imported successfully")
print(f"Scikit-learn and imbalanced-learn ready.")

All libraries imported successfully
Scikit-learn and imbalanced-learn ready.


In [18]:
# ============================================================
# STEP 2: Load Data
# ============================================================

# Update this path to your actual file location
DATA_PATH = "CICIDS2017_preprocessed.csv"

print(" Loading dataset...")
df = pd.read_csv(DATA_PATH)
print(f" Loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")

# Separate features and label
X = df.drop(columns=['Label'])
y = df['Label']

print(f"\n  Class distribution:")
class_counts = y.value_counts().sort_index()
for cls, count in class_counts.items():
    pct = count / len(y) * 100
    print(f"   Class {cls:>2}: {count:>8,} samples ({pct:.4f}%)")

 Loading dataset...
 Loaded: 756,208 rows × 67 columns

  Class distribution:
   Class  0:  628,486 samples (83.1102%)
   Class  1:      584 samples (0.0772%)
   Class  2:   38,404 samples (5.0785%)
   Class  3:    3,086 samples (0.4081%)
   Class  4:   51,854 samples (6.8571%)
   Class  5:    1,568 samples (0.2074%)
   Class  6:    1,616 samples (0.2137%)
   Class  7:    1,779 samples (0.2353%)
   Class  8:        3 samples (0.0004%)
   Class  9:       11 samples (0.0015%)
   Class 10:   27,208 samples (3.5980%)
   Class 11:      966 samples (0.1277%)
   Class 12:      441 samples (0.0583%)
   Class 13:        6 samples (0.0008%)
   Class 14:      196 samples (0.0259%)


In [19]:
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.utils import resample

X = df.drop(columns=['Label'])
y = df['Label']

# Full CV splitter (used for final evaluation)
skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

# Small stratified sample ONLY for GridSearchCV (speed)
X_gs, _, y_gs, _ = train_test_split(
    X, y, train_size=50000, stratify=y, random_state=42
)
skf_gs = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

print(f"✅ Grid search sample: {X_gs.shape[0]:,} rows (stratified)")
print(f"✅ Full CV will run on all {len(X):,} rows after best params found")

✅ Grid search sample: 50,000 rows (stratified)
✅ Full CV will run on all 756,208 rows after best params found


In [22]:
# ============================================================
# STEP 4: Cross-Validation Evaluation Helper
# Reports mean ± std across 3 folds (catches overfitting)
# ============================================================
import warnings
warnings.filterwarnings('ignore')

results = []

def evaluate_model_cv(name, model, X, y, cv):
    print(f"\n{'='*60}")
    print(f"  🚀 Cross-Validating: {name}")
    print(f"{'='*60}")

    fold_metrics = {
        'acc': [], 'prec': [], 'rec': [],
        'f1_mac': [], 'f1_wt': [], 'mcc': []
    }
    fold_times = []
    all_preds = np.zeros(len(y), dtype=int)   # store OOF predictions

    for fold, (train_idx, test_idx) in enumerate(cv.split(X, y), 1):
        X_tr, X_te = X.iloc[train_idx], X.iloc[test_idx]
        y_tr, y_te = y.iloc[train_idx], y.iloc[test_idx]

        start = time.time()
        model.fit(X_tr, y_tr)
        elapsed = round(time.time() - start, 2)
        fold_times.append(elapsed)

        y_pred = model.predict(X_te)
        all_preds[test_idx] = y_pred

        fold_metrics['acc'].append(accuracy_score(y_te, y_pred) * 100)
        fold_metrics['prec'].append(precision_score(y_te, y_pred, average='macro', zero_division=0) * 100)
        fold_metrics['rec'].append(recall_score(y_te, y_pred, average='macro', zero_division=0) * 100)
        fold_metrics['f1_mac'].append(f1_score(y_te, y_pred, average='macro', zero_division=0) * 100)
        fold_metrics['f1_wt'].append(f1_score(y_te, y_pred, average='weighted', zero_division=0) * 100)
        fold_metrics['mcc'].append(matthews_corrcoef(y_te, y_pred))

        print(f"  Fold {fold}: Acc={fold_metrics['acc'][-1]:.2f}%  "
              f"F1-Mac={fold_metrics['f1_mac'][-1]:.2f}%  "
              f"MCC={fold_metrics['mcc'][-1]:.4f}  ({elapsed}s)")

    # Compute mean ± std
    def ms(lst): return np.mean(lst), np.std(lst)

    acc_m,   acc_s   = ms(fold_metrics['acc'])
    prec_m,  prec_s  = ms(fold_metrics['prec'])
    rec_m,   rec_s   = ms(fold_metrics['rec'])
    f1m_m,   f1m_s   = ms(fold_metrics['f1_mac'])
    f1w_m,   f1w_s   = ms(fold_metrics['f1_wt'])
    mcc_m,   mcc_s   = ms(fold_metrics['mcc'])

    print(f"\n  📊 MEAN  ± STD across 3 folds:")
    print(f"     Accuracy:       {acc_m:.2f}% ± {acc_s:.2f}%")
    print(f"     Precision Mac:  {prec_m:.2f}% ± {prec_s:.2f}%")
    print(f"     Recall Mac:     {rec_m:.2f}% ± {rec_s:.2f}%")
    print(f"     F1-Macro:       {f1m_m:.2f}% ± {f1m_s:.2f}%  ← KEY")
    print(f"     F1-Weighted:    {f1w_m:.2f}% ± {f1w_s:.2f}%")
    print(f"     MCC:            {mcc_m:.4f} ± {mcc_s:.4f}")
    print(f"     Avg Train Time: {np.mean(fold_times):.1f}s/fold")

    # High std = overfitting; flag it
    if f1m_s > 5:
        print(f"  ⚠️  High F1-Macro std ({f1m_s:.2f}%) — model is unstable across folds")
    else:
        print(f"  ✅ Stable across folds (std={f1m_s:.2f}%)")

    results.append({
        'Model':             name,
        'Accuracy':          round(acc_m, 2),
        'Acc Std':           round(acc_s, 2),
        'Precision (Macro)': round(prec_m, 2),
        'Recall (Macro)':    round(rec_m, 2),
        'F1-Macro':          round(f1m_m, 2),
        'F1-Macro Std':      round(f1m_s, 2),
        'F1-Weighted':       round(f1w_m, 2),
        'MCC':               round(mcc_m, 4),
        'MCC Std':           round(mcc_s, 4),
        'Avg Train Time (s)':round(np.mean(fold_times), 1),
    })

    return model, all_preds   # all_preds = out-of-fold predictions

print("✅ CV evaluation helper ready")

✅ CV evaluation helper ready


In [23]:
# ============================================================
# MODEL 1: Random Forest — GridSearchCV
# ============================================================
from sklearn.ensemble import RandomForestClassifier

rf_grid = {
    'n_estimators':    [100, 200],
    'max_depth':       [15, 25, None],
    'min_samples_leaf':[2, 5],
    'max_features':    ['sqrt', 'log2'],
}

rf_gs = GridSearchCV(
    RandomForestClassifier(class_weight='balanced', random_state=42, n_jobs=-1),
    rf_grid,
    cv=skf_gs,
    scoring='f1_macro',
    n_jobs=-1,
    verbose=1
)
rf_gs.fit(X_gs, y_gs)

print(f"\n✅ Best RF params:  {rf_gs.best_params_}")
print(f"   Best CV F1-Mac (sample): {rf_gs.best_score_*100:.2f}%")

# Retrain best model on full data with 3-fold CV
rf_best = RandomForestClassifier(
    **rf_gs.best_params_,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)
rf_model, rf_preds = evaluate_model_cv(
    "Random Forest (Tuned)", rf_best, X, y, skf
)

Fitting 3 folds for each of 24 candidates, totalling 72 fits

✅ Best RF params:  {'max_depth': 25, 'max_features': 'sqrt', 'min_samples_leaf': 5, 'n_estimators': 100}
   Best CV F1-Mac (sample): 85.15%

  🚀 Cross-Validating: Random Forest (Tuned)
  Fold 1: Acc=99.83%  F1-Mac=73.65%  MCC=0.9944  (41.75s)
  Fold 2: Acc=99.72%  F1-Mac=82.89%  MCC=0.9907  (40.3s)
  Fold 3: Acc=99.79%  F1-Mac=80.92%  MCC=0.9932  (41.13s)

  📊 MEAN  ± STD across 3 folds:
     Accuracy:       99.78% ± 0.05%
     Precision Mac:  78.76% ± 4.23%
     Recall Mac:     81.59% ± 4.77%
     F1-Macro:       79.15% ± 3.98%  ← KEY
     F1-Weighted:    99.80% ± 0.04%
     MCC:            0.9928 ± 0.0016
     Avg Train Time: 41.1s/fold
  ✅ Stable across folds (std=3.98%)


In [25]:
brf_gs = GridSearchCV(
    BalancedRandomForestClassifier(
        sampling_strategy='auto',        # ← 'auto' works on any size sample
        replacement=True,
        class_weight='balanced_subsample',
        random_state=42,
        n_jobs=-1
    ),
    brf_grid,
    cv=skf_gs,
    scoring='f1_macro',
    n_jobs=1,
    verbose=1
)
brf_gs.fit(X_gs, y_gs)

print(f"\n✅ Best BRF params: {brf_gs.best_params_}")
print(f"   Best CV F1-Mac (sample): {brf_gs.best_score_*100:.2f}%")

brf_best = BalancedRandomForestClassifier(
    **brf_gs.best_params_,
    sampling_strategy={0: 60000},        # ← dict only here, full dataset has enough BENIGN
    replacement=True,
    class_weight='balanced_subsample',
    random_state=42,
    n_jobs=-1
)
brf_model, brf_preds = evaluate_model_cv(
    "Balanced Random Forest (Tuned)", brf_best, X, y, skf
)

Fitting 3 folds for each of 16 candidates, totalling 48 fits

✅ Best BRF params: {'max_depth': 15, 'max_features': 'log2', 'min_samples_leaf': 2, 'n_estimators': 100}
   Best CV F1-Mac (sample): 13.79%

  🚀 Cross-Validating: Balanced Random Forest (Tuned)
  Fold 1: Acc=98.29%  F1-Mac=63.21%  MCC=0.9468  (18.69s)
  Fold 2: Acc=97.52%  F1-Mac=72.37%  MCC=0.9254  (18.18s)
  Fold 3: Acc=98.23%  F1-Mac=70.95%  MCC=0.9452  (17.53s)

  📊 MEAN  ± STD across 3 folds:
     Accuracy:       98.01% ± 0.35%
     Precision Mac:  66.84% ± 4.36%
     Recall Mac:     82.32% ± 4.91%
     F1-Macro:       68.85% ± 4.02%  ← KEY
     F1-Weighted:    98.62% ± 0.21%
     MCC:            0.9391 ± 0.0097
     Avg Train Time: 18.1s/fold
  ✅ Stable across folds (std=4.02%)


In [26]:
# ============================================================
# MODEL 3: BaggingClassifier — GridSearchCV
# ============================================================
from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier

bag_grid = {
    'n_estimators': [50, 100],
    'max_samples':  [0.5, 0.7],
    'max_features': [0.7, 0.9],
    'estimator__max_depth':       [15, 25],
    'estimator__min_samples_leaf':[2, 5],
}

bag_gs = GridSearchCV(
    BaggingClassifier(
        estimator=DecisionTreeClassifier(class_weight='balanced', random_state=42),
        bootstrap=True,
        n_jobs=1,       # ← must stay 1 (memory)
        random_state=42
    ),
    bag_grid,
    cv=skf_gs,
    scoring='f1_macro',
    n_jobs=1,
    verbose=1
)
bag_gs.fit(X_gs, y_gs)

print(f"\n✅ Best Bagging params: {bag_gs.best_params_}")
print(f"   Best CV F1-Mac (sample): {bag_gs.best_score_*100:.2f}%")

# Rebuild with best params (keep n_jobs=1 to avoid memory crash)
best_dt = DecisionTreeClassifier(
    max_depth=bag_gs.best_params_['estimator__max_depth'],
    min_samples_leaf=bag_gs.best_params_['estimator__min_samples_leaf'],
    class_weight='balanced',
    random_state=42
)
bag_best = BaggingClassifier(
    estimator=best_dt,
    n_estimators=bag_gs.best_params_['n_estimators'],
    max_samples=bag_gs.best_params_['max_samples'],
    max_features=bag_gs.best_params_['max_features'],
    bootstrap=True,
    n_jobs=1,
    random_state=42
)
bag_model, bag_preds = evaluate_model_cv(
    "BaggingClassifier (Tuned)", bag_best, X, y, skf
)

Fitting 3 folds for each of 32 candidates, totalling 96 fits

✅ Best Bagging params: {'estimator__max_depth': 25, 'estimator__min_samples_leaf': 2, 'max_features': 0.9, 'max_samples': 0.7, 'n_estimators': 100}
   Best CV F1-Mac (sample): 85.58%

  🚀 Cross-Validating: BaggingClassifier (Tuned)
  Fold 1: Acc=99.88%  F1-Mac=79.78%  MCC=0.9959  (1263.42s)
  Fold 2: Acc=99.87%  F1-Mac=85.50%  MCC=0.9958  (1221.9s)
  Fold 3: Acc=99.87%  F1-Mac=81.19%  MCC=0.9957  (1216.33s)

  📊 MEAN  ± STD across 3 folds:
     Accuracy:       99.87% ± 0.00%
     Precision Mac:  83.02% ± 2.71%
     Recall Mac:     82.56% ± 2.16%
     F1-Macro:       82.16% ± 2.44%  ← KEY
     F1-Weighted:    99.87% ± 0.00%
     MCC:            0.9958 ± 0.0001
     Avg Train Time: 1233.9s/fold
  ✅ Stable across folds (std=2.44%)


In [33]:
# ============================================================
# MODEL 4: EasyEnsemble — reduced GridSearchCV
# ============================================================

# Use even smaller sample — EE is too slow for 50k
X_ee, _, y_ee, _ = train_test_split(
    X, y, train_size=10000, stratify=y, random_state=42
)
skf_ee = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

ee_grid = {
    'n_estimators': [5, 10],    # ← only 2 options, 6 fits total
}

ee_gs = GridSearchCV(
    EasyEnsembleClassifier(random_state=42, n_jobs=1),
    ee_grid,
    cv=skf_ee,
    scoring='f1_macro',
    n_jobs=1,
    verbose=1
)
ee_gs.fit(X_ee, y_ee)

print(f"\n✅ Best EE params: {ee_gs.best_params_}")
print(f"   Best CV F1-Mac (sample): {ee_gs.best_score_*100:.2f}%")

ee_best = EasyEnsembleClassifier(
    n_estimators=ee_gs.best_params_['n_estimators'],
    random_state=42,
    n_jobs=1               # ← 1 to avoid memory issues on full CV
)
ee_model, ee_preds = evaluate_model_cv(
    "EasyEnsemble (Tuned)", ee_best, X, y, skf
)

Fitting 3 folds for each of 2 candidates, totalling 6 fits

✅ Best EE params: {'n_estimators': 10}
   Best CV F1-Mac (sample): 18.48%

  🚀 Cross-Validating: EasyEnsemble (Tuned)
  Fold 1: Acc=74.65%  F1-Mac=24.79%  MCC=0.3505  (2.39s)
  Fold 2: Acc=6.54%  F1-Mac=17.59%  MCC=0.1586  (2.35s)
  Fold 3: Acc=64.64%  F1-Mac=25.70%  MCC=0.1041  (2.44s)

  📊 MEAN  ± STD across 3 folds:
     Accuracy:       48.61% ± 30.03%
     Precision Mac:  34.15% ± 0.75%
     Recall Mac:     31.79% ± 2.36%
     F1-Macro:       22.69% ± 3.63%  ← KEY
     F1-Weighted:    53.10% ± 30.71%
     MCC:            0.2044 ± 0.1057
     Avg Train Time: 2.4s/fold
  ✅ Stable across folds (std=3.63%)


In [32]:
# ============================================================
# MODEL 5: RUSBoost — GridSearchCV
# ============================================================
from imblearn.ensemble import RUSBoostClassifier

rus_grid = {
    'n_estimators': [100, 200],
    'learning_rate':[0.05, 0.1, 0.5],
}

rus_gs = GridSearchCV(
    RUSBoostClassifier(sampling_strategy='auto', random_state=42),
    rus_grid,
    cv=skf_gs,
    scoring='f1_macro',
    n_jobs=-1,
    verbose=1
)
rus_gs.fit(X_gs, y_gs)

print(f"\n✅ Best RUSBoost params: {rus_gs.best_params_}")
print(f"   Best CV F1-Mac (sample): {rus_gs.best_score_*100:.2f}%")

rus_best = RUSBoostClassifier(
    **rus_gs.best_params_,
    sampling_strategy='auto',
    random_state=42
)
rus_model, rus_preds = evaluate_model_cv(
    "RUSBoost (Tuned)", rus_best, X, y, skf
)

Fitting 3 folds for each of 6 candidates, totalling 18 fits

✅ Best RUSBoost params: {'learning_rate': 0.5, 'n_estimators': 100}
   Best CV F1-Mac (sample): 11.03%

  🚀 Cross-Validating: RUSBoost (Tuned)
  Fold 1: Acc=83.12%  F1-Mac=10.30%  MCC=0.0783  (0.4s)
  Fold 2: Acc=82.83%  F1-Mac=6.10%  MCC=0.1222  (0.75s)
  Fold 3: Acc=82.88%  F1-Mac=6.10%  MCC=0.1255  (0.4s)

  📊 MEAN  ± STD across 3 folds:
     Accuracy:       82.94% ± 0.13%
     Precision Mac:  6.64% ± 1.43%
     Recall Mac:     13.31% ± 0.00%
     F1-Macro:       7.50% ± 1.98%  ← KEY
     F1-Weighted:    75.86% ± 0.18%
     MCC:            0.1086 ± 0.0215
     Avg Train Time: 0.5s/fold
  ✅ Stable across folds (std=1.98%)


In [35]:
results_df = pd.DataFrame(results).set_index('Model')

# Also save best params found per model
best_params_log = {
    "Random Forest":      rf_gs.best_params_,
    "Balanced RF":        brf_gs.best_params_,
    "BaggingClassifier":  bag_gs.best_params_,
    "EasyEnsemble":       ee_gs.best_params_,
    "RUSBoost":           rus_gs.best_params_,
}
esults_df = pd.DataFrame(results).set_index('Model')
results_df = results_df[~results_df.index.duplicated(keep='first')]  

print("\n" + "="*70)
print("  CASE 7 — GRIDSEARCH TUNED RESULTS (3-Fold CV, mean ± std)")
print("="*70)
display(results_df)

print("\n📋 Best hyperparameters found:")
for model, params in best_params_log.items():
    print(f"\n  {model}: {params}")

results_df.to_csv("Case7_Results_GridSearchCV.csv")
pd.DataFrame(best_params_log).T.to_csv("Case7_BestParams.csv")
print("\n✅ Saved Case7_Results_GridSearchCV.csv and Case7_BestParams.csv")


  CASE 7 — GRIDSEARCH TUNED RESULTS (3-Fold CV, mean ± std)


,Accuracy,Acc Std,Precision (Macro),Recall (Macro),F1-Macro,F1-Macro Std,F1-Weighted,MCC,MCC Std,Avg Train Time (s)
Model,,,,,,,,,,
Random Forest (Tuned),99.78,0.05,78.76,81.59,79.15,3.98,99.80,0.9928,0.0016,41.1
Balanced Random Forest (Tuned),98.01,0.35,66.84,82.32,68.85,4.02,98.62,0.9391,0.0097,18.1
BaggingClassifier (Tuned),99.87,0.00,83.02,82.56,82.16,2.44,99.87,0.9958,0.0001,1233.9
EasyEnsemble (Tuned),48.61,30.03,34.15,31.79,22.69,3.63,53.10,0.2044,0.1057,1.7
RUSBoost (Tuned),82.94,0.13,6.64,13.31,7.50,1.98,75.86,0.1086,0.0215,0.6



📋 Best hyperparameters found:

  Random Forest: {'max_depth': 25, 'max_features': 'sqrt', 'min_samples_leaf': 5, 'n_estimators': 100}

  Balanced RF: {'max_depth': 15, 'max_features': 'log2', 'min_samples_leaf': 2, 'n_estimators': 100}

  BaggingClassifier: {'estimator__max_depth': 25, 'estimator__min_samples_leaf': 2, 'max_features': 0.9, 'max_samples': 0.7, 'n_estimators': 100}

  EasyEnsemble: {'n_estimators': 10}

  RUSBoost: {'learning_rate': 0.5, 'n_estimators': 100}

✅ Saved Case7_Results_GridSearchCV.csv and Case7_BestParams.csv


In [36]:
# ============================================================
# STEP 11: Side-by-side comparison with Case 1
# ============================================================

case1_data = {
    'Model': ['Naive Bayes', 'KNN', 'Decision Tree',
              'Random Forest', 'XGBoost', 'Stacked Ensemble'],
    'Accuracy':  [6.83,  98.59, 99.80, 99.81, 99.25, 99.83],
    'F1-Macro':  [10.19, 67.80, 81.94, 79.29, 54.38, 71.83],
    'MCC':       [0.1653, 0.9537, 0.9935, 0.9937, 0.9750, 0.9943],
    'Case': ['Case 1'] * 6
}
case1_df = pd.DataFrame(case1_data).set_index('Model')

case7_compare = results_df[['Accuracy', 'F1-Macro', 'MCC']].copy()
case7_compare['Case'] = 'Case 7'

compare_df = pd.concat([case1_df, case7_compare])

print("\n CASE 1 vs CASE 7 — F1-Macro Comparison")
print("="*55)
print(compare_df[['Accuracy', 'F1-Macro', 'MCC', 'Case']].to_string())

# Highlight best F1-Macro in Case 7
best_case7 = results_df['F1-Macro'].max()
best_model  = results_df['F1-Macro'].idxmax()
rf_f1 = 79.29  # from Case 1

print(f"\n Best Case 7 Model:  {best_model}")
print(f"   F1-Macro: {best_case7}%  vs  RF Baseline: {rf_f1}%")
improvement = best_case7 - rf_f1
if improvement > 0:
    print(f"   IMPROVEMENT of +{improvement:.2f}% over standard Random Forest")
else:
    print(f"    No F1-Macro improvement ({improvement:.2f}%) — analyze why")


 CASE 1 vs CASE 7 — F1-Macro Comparison
                                Accuracy  F1-Macro     MCC    Case
Model                                                             
Naive Bayes                         6.83     10.19  0.1653  Case 1
KNN                                98.59     67.80  0.9537  Case 1
Decision Tree                      99.80     81.94  0.9935  Case 1
Random Forest                      99.81     79.29  0.9937  Case 1
XGBoost                            99.25     54.38  0.9750  Case 1
Stacked Ensemble                   99.83     71.83  0.9943  Case 1
Random Forest (Tuned)              99.78     79.15  0.9928  Case 7
Balanced Random Forest (Tuned)     98.01     68.85  0.9391  Case 7
BaggingClassifier (Tuned)          99.87     82.16  0.9958  Case 7
EasyEnsemble (Tuned)               48.61     22.69  0.2044  Case 7
RUSBoost (Tuned)                   82.94      7.50  0.1086  Case 7

 Best Case 7 Model:  BaggingClassifier (Tuned)
   F1-Macro: 82.16%  vs  RF Baseline: 79

In [39]:
# ============================================================
# STEP 12: Final Consolidated Results Table (report-ready)
# ============================================================

case1_full = pd.DataFrame({
    'Model':              ['Naive Bayes','KNN','Decision Tree',
                           'Random Forest','XGBoost','Stacked Ensemble'],
    'Accuracy':           [6.83, 98.59, 99.80, 99.81, 99.25, 99.83],
    'Precision (Macro)':  [15.63, 68.49, 82.0,  80.18, 54.05, 72.59],
    'Recall (Macro)':     [34.94, 67.33, 81.89, 78.51, 54.94, 71.17],
    'F1-Macro':           [10.19, 67.80, 81.94, 79.29, 54.38, 71.83],
    'F1-Weighted':        [6.54,  98.59, 99.80, 99.81, 99.26, 99.83],
    'MCC':                [0.1653,0.9537,0.9935,0.9937,0.9750,0.9943],
    'Case':               ['Case 1']*6
}).set_index('Model')

case7_full = results_df.copy()
case7_full['Case'] = 'Case 7'

combined = pd.concat([case1_full, case7_full])
combined.to_csv("Combined_Case1_Case7_Results.csv")

print("\n COMBINED RESULTS — CASE 1 + CASE 7")
print("="*90)
print(combined[['Case','Accuracy','F1-Macro','F1-Weighted','MCC']].to_string())
print("\n Saved to Combined_Case1_Case7_Results.csv")


 COMBINED RESULTS — CASE 1 + CASE 7
                                  Case  Accuracy  F1-Macro  F1-Weighted     MCC
Model                                                                          
Naive Bayes                     Case 1      6.83     10.19         6.54  0.1653
KNN                             Case 1     98.59     67.80        98.59  0.9537
Decision Tree                   Case 1     99.80     81.94        99.80  0.9935
Random Forest                   Case 1     99.81     79.29        99.81  0.9937
XGBoost                         Case 1     99.25     54.38        99.26  0.9750
Stacked Ensemble                Case 1     99.83     71.83        99.83  0.9943
Random Forest (Tuned)           Case 7     99.78     79.15        99.80  0.9928
Balanced Random Forest (Tuned)  Case 7     98.01     68.85        98.62  0.9391
BaggingClassifier (Tuned)       Case 7     99.87     82.16        99.87  0.9958
EasyEnsemble (Tuned)            Case 7     48.61     22.69        53.10  0.2044
RUS